# FraudShield AI — Exploratory Data Analysis (EDA)

**Objectif :** Analyser le dataset de transactions synthétiques pour comprendre la structure, les distributions, les corrélations et les problèmes potentiels avant la phase de modélisation.

**Plan :**
1. Chargement et aperçu des données
2. Analyse de la variable cible (`is_fraud`)
3. Distribution des variables numériques
4. Analyse des variables catégorielles
5. Corrélations et relations avec la cible
6. Détection des valeurs aberrantes (outliers)
7. Problèmes identifiés et recommandations

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid', palette='husl')

DATA_PATH = Path('../ml/data/raw/transactions.csv')

## 1. Chargement et aperçu des données

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
df.head()

In [ ]:
# Types et valeurs manquantes
info = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.notnull().sum(),
    'null': df.isnull().sum(),
    'null_%': (df.isnull().mean() * 100).round(2),
    'n_unique': df.nunique(),
})
print('=== Profil des colonnes ===')
info

In [ ]:
# Statistiques descriptives des variables numériques
df.describe().T.style.background_gradient(cmap='Blues')

## 2. Analyse de la variable cible — déséquilibre des classes

> **Problème identifié :** Le dataset est fortement déséquilibré (~2 % de fraudes). Ce déséquilibre nécessite une stratégie adaptée : `stratify=y` lors du split, et SMOTE en oversampling.

In [ ]:
fraud_counts = df['is_fraud'].value_counts()
fraud_pct = df['is_fraud'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(['Légitime (0)', 'Fraude (1)'], fraud_counts.values,
            color=['#2196F3', '#F44336'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribution de la variable cible', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Nombre de transactions')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 50, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=11)

# Pie chart
axes[1].pie(fraud_counts.values, labels=['Légitime', 'Fraude'],
            colors=['#2196F3', '#F44336'], autopct='%1.1f%%',
            startangle=90, explode=(0, 0.1))
axes[1].set_title(f'Taux de fraude : {fraud_pct:.2f}%', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()
print(f'Classe 0 (Légitime) : {fraud_counts[0]:,} ({100-fraud_pct:.2f}%)')
print(f'Classe 1 (Fraude)   : {fraud_counts[1]:,} ({fraud_pct:.2f}%)')
print(f'Ratio déséquilibre  : 1:{fraud_counts[0]//fraud_counts[1]}')

## 3. Distribution des variables numériques

In [ ]:
numeric_cols = ['amount', 'hour_of_day', 'day_of_week', 'transaction_count_1h',
                'transaction_count_24h', 'amount_mean_1h', 'amount_std_1h',
                'merchant_risk_score', 'distance_from_home', 'velocity_score']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    for label, color in [(0, '#2196F3'), (1, '#F44336')]:
        axes[i].hist(df[df['is_fraud'] == label][col], bins=40,
                     alpha=0.6, color=color,
                     label='Légitime' if label == 0 else 'Fraude',
                     density=True)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=8)

plt.suptitle('Distributions des variables numériques (Légitimes vs Fraudes)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution du montant (log scale) — variable clé
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [(0, '#2196F3', 'Légitime'), (1, '#F44336', 'Fraude')]:
    subset = df[df['is_fraud'] == label]['amount']
    axes[0].hist(subset, bins=60, alpha=0.6, color=color, label=name, density=True)
    axes[1].hist(np.log1p(subset), bins=60, alpha=0.6, color=color, label=name, density=True)

axes[0].set_title('Distribution du montant (échelle originale)', fontweight='bold')
axes[0].set_xlabel('Montant (€)')
axes[1].set_title('Distribution du montant (log1p)', fontweight='bold')
axes[1].set_xlabel('log(1 + montant)')
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

print('Statistiques du montant par classe :')
df.groupby('is_fraud')['amount'].describe().round(2)

## 4. Analyse des variables catégorielles

In [ ]:
cat_cols = ['merchant_category', 'card_type', 'entry_mode']
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, col in zip(axes, cat_cols):
    fraud_rate = df.groupby(col)['is_fraud'].mean().sort_values(ascending=False)
    bars = ax.bar(fraud_rate.index, fraud_rate.values * 100,
                  color=plt.cm.RdYlGn_r(fraud_rate.values / fraud_rate.max()))
    ax.set_title(f'Taux de fraude par {col}', fontweight='bold')
    ax.set_ylabel('Taux de fraude (%)')
    ax.tick_params(axis='x', rotation=45)
    for bar, val in zip(bars, fraud_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9)

plt.suptitle('Taux de fraude par variable catégorielle', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Analyse temporelle : fraudes par heure de la journée
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

hourly = df.groupby('hour_of_day')['is_fraud'].agg(['sum', 'count', 'mean'])
axes[0].bar(hourly.index, hourly['mean'] * 100, color='#F44336', alpha=0.8)
axes[0].set_title('Taux de fraude par heure de la journée', fontweight='bold')
axes[0].set_xlabel('Heure')
axes[0].set_ylabel('Taux de fraude (%)')
axes[0].axhline(df['is_fraud'].mean() * 100, color='blue', linestyle='--',
                label=f'Moyenne globale ({df["is_fraud"].mean()*100:.2f}%)')
axes[0].legend()

daily = df.groupby('day_of_week')['is_fraud'].mean()
day_names = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']
axes[1].bar([day_names[i] for i in daily.index], daily.values * 100,
            color='#FF9800', alpha=0.8)
axes[1].set_title('Taux de fraude par jour de la semaine', fontweight='bold')
axes[1].set_xlabel('Jour')
axes[1].set_ylabel('Taux de fraude (%)')

plt.tight_layout()
plt.show()

## 5. Corrélations et relations avec la cible

In [ ]:
# Matrice de corrélation (variables numériques)
numeric_df = df[numeric_cols + ['is_fraud']].copy()
corr_matrix = numeric_df.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Matrice de corrélation — Variables numériques', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Corrélations avec is_fraud — classement
target_corr = corr_matrix['is_fraud'].drop('is_fraud').sort_values(key=abs, ascending=False)
colors = ['#F44336' if v > 0 else '#2196F3' for v in target_corr.values]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(target_corr.index, target_corr.values, color=colors, alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Corrélation des features avec is_fraud', fontsize=14, fontweight='bold')
ax.set_xlabel('Coefficient de corrélation de Pearson')
for bar, val in zip(bars, target_corr.values):
    x = bar.get_width() + 0.005 if val >= 0 else bar.get_width() - 0.005
    ax.text(x, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.show()

print('Top 5 features les plus corrélées avec is_fraud :')
print(target_corr.head())

## 6. Détection des valeurs aberrantes (Outliers)

> **Remarque :** Les outliers sur `amount` peuvent être légitimes (grosses transactions) ou frauduleux. On utilise le Z-score pour les identifier.

In [ ]:
# Box plots pour détecter les outliers
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    data_legit = df[df['is_fraud'] == 0][col]
    data_fraud = df[df['is_fraud'] == 1][col]
    axes[i].boxplot([data_legit, data_fraud],
                    labels=['Légitime', 'Fraude'],
                    patch_artist=True,
                    boxprops=dict(facecolor='#2196F3', alpha=0.6),
                    medianprops=dict(color='black', linewidth=2))
    axes[i].set_title(col, fontsize=10, fontweight='bold')

plt.suptitle('Box plots — Comparaison Légitimes vs Fraudes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Z-score outliers sur le montant
z_scores = np.abs(stats.zscore(df['amount']))
outlier_threshold = 3.0
outliers = df[z_scores > outlier_threshold]

print(f'Outliers détectés (|Z| > {outlier_threshold}) : {len(outliers):,} ({len(outliers)/len(df)*100:.2f}%)')
print(f"Taux de fraude dans les outliers : {outliers['is_fraud'].mean()*100:.2f}%")
print(f"Taux de fraude global            : {df['is_fraud'].mean()*100:.2f}%")
print(f"\n→ Les outliers de montant ont {outliers['is_fraud'].mean()/df['is_fraud'].mean():.1f}x "
      f"plus de risque de fraude que la moyenne.")

## 7. Problèmes identifiés et recommandations

| # | Problème identifié | Sévérité | Solution appliquée dans le pipeline |
|---|---|---|---|
| 1 | **Déséquilibre des classes** (~2% fraudes) | Haute | SMOTE oversampling + `stratify=y` |
| 2 | **Distribution asymétrique de `amount`** | Moyenne | Transformation `log1p` (feature engineering) |
| 3 | **Variables temporelles cycliques** (`hour_of_day`, `day_of_week`) | Moyenne | Encodage sin/cos |
| 4 | **Variables catégorielles** (`merchant_category`, etc.) | Faible | One-hot encoding |
| 5 | **Outliers sur `amount`** | Faible | StandardScaler robuste, pas de suppression (peuvent être des fraudes) |
| 6 | **Risque de data leakage** | Haute | `get_feature_columns()` exclut explicitement `transaction_id`, `customer_id`, `is_fraud` |

In [ ]:
# Résumé final
print('=== RÉSUMÉ EDA ===')
print(f'Dataset         : {len(df):,} transactions')
print(f'Période         : {df["timestamp"].min()} → {df["timestamp"].max()}')
print(f'Fraudes         : {df["is_fraud"].sum():,} ({df["is_fraud"].mean()*100:.2f}%)')
print(f'Valeurs nulles  : {df.isnull().sum().sum()}')
print(f'Doublons        : {df.duplicated("transaction_id").sum()}')
print(f'Features num.   : {len(numeric_cols)}')
print(f'Features cat.   : {len(cat_cols)}')
print(f'\nFeature la plus corrélée avec is_fraud : {target_corr.abs().idxmax()} ({target_corr.abs().max():.3f})')